# 02 - Data Validation (Phase 3)

Domain-specific validation on top of the structural audit from Phase 2
(`01_data_audit.ipynb`). All logic lives in `data_contracts/` and
`src/promolift/validation/` -- this notebook only calls into it and
reports results.

Two independent checks:
- **Referential integrity** -- is the raw data itself trustworthy? (schema
  contracts, client/product foreign keys, transaction date integrity)
- **Experiment validity** -- is the randomized experiment behind
  `uplift_train.csv` trustworthy? (treatment/control balance, outcome
  distribution, a raw ATE sanity check)

If either of these fails, no causal model built later can be trusted --
this notebook is the gate before Phase 5+ (modeling).

In [ ]:
import pandera.errors

from promolift.data.loader import Dataset, load_lazy
from promolift.features.demographics import build_demographic_features
from promolift.validation.experiment_validity import (
    categorical_covariate_balance,
    numeric_covariate_balance,
    outcome_distribution,
    propensity_check,
    raw_ate_sanity_check,
)
from promolift.validation.referential_integrity import (
    client_foreign_key_integrity,
    product_foreign_key_integrity,
    transaction_date_integrity,
)
from promolift.validation.schema_validation import validate_dataset

## 1. Schema contracts

Validates every small file eagerly (full dtype + value checks) and
`purchases.csv` lazily (dtype checks only -- see
`schema_validation.validate_dataset` docstring for why value-level checks
aren't reliable on a Polars `LazyFrame`).

In [ ]:
for dataset in Dataset:
    try:
        result = validate_dataset(dataset)
        if dataset is Dataset.PURCHASES:
            result.collect()
        print(f"{dataset.value}: PASSED")
    except pandera.errors.SchemaError as e:
        print(f"{dataset.value}: FAILED\n{e}\n")

`clients` is expected to fail here: 313 of 400,162 rows fall outside the
schema's `[0, 120]` bound (a structural "physically possible" sanity check,
min -7491, max 1901). This is a known, real data-quality issue.

Feature engineering (Phase 5) applies a *tighter*, business-appropriate bound
of `[10, 100]` for a retail loyalty program shopper specifically, flagging
1,404 rows -- the two thresholds are intentionally different: this schema
checks physical plausibility, `demographics.build_demographic_features`
checks business plausibility.

## 2. Referential integrity

In [ ]:
print(client_foreign_key_integrity())
print(product_foreign_key_integrity())
print(transaction_date_integrity())

## 3. Experiment validity

Checks whether the treatment/control split in `uplift_train.csv` behaves
like a genuine randomization, and whether the observed effect is real.

In [ ]:
print(numeric_covariate_balance("age"))
print(categorical_covariate_balance("gender"))
print(outcome_distribution())
print(raw_ate_sanity_check())

## 4. Propensity and overlap check

Fits a classifier to check whether treatment is predictable from
covariates -- under true randomization it shouldn't be (AUC ~ 0.5).
Also checks propensity-score overlap between arms (positivity), a core
assumption for causal identification.

In [ ]:
demo = build_demographic_features(transaction_date_integrity().max_date).collect()
joined = load_lazy(Dataset.UPLIFT_TRAIN).collect().join(demo, on="client_id", how="left")

report = propensity_check(joined.select("age", "gender"), joined["treatment_flg"])
print(report)

## Phase 3 summary

- **Referential integrity: clean.** 0 orphaned `client_id`s or `product_id`s
  in `purchases.csv`; all `transaction_datetime` values parse and fall in a
  sane range (Nov 2018 - Mar 2019), no future dates.
- **Experiment validity: clean.** Age (SMD ~0.005) and gender (max
  proportion diff ~0.002) are well balanced between arms, far under the 0.1
  rule-of-thumb threshold. The raw ATE is +3.3pp with a 95% CI that excludes
  zero -- a real, non-degenerate effect.
- **Propensity check: clean.** AUC ~0.50 -- treatment is not predictable
  from age/gender, as expected under true randomization. Propensity-score
  ranges overlap widely between arms (positivity holds).
- **Two known, real bugs found via cross-validation against an independent
  implementation, both fixed in `demographics.py`:** post-treatment
  redemption-date leakage (44,953 clients redeemed after the treatment
  window -- now excluded from `has_redeemed`), and a too-permissive age
  bound (tightened to `[10, 100]` for feature engineering; the schema above
  intentionally keeps the wider `[0, 120]` structural bound).

**Conclusion:** the dataset and the experiment behind it are trustworthy
enough to build causal models on top of (Phase 5+).